# Final model: 2x2D MaxViT + 3D ResNeXt + CrossGate
Architecture remains in `scripts/final_model.py`; recovery, caching and reporting live in `scripts/final_training.py`.

**Recovery contract:** `last.pt` is committed only after a complete optimizer/scaler/scheduler operation with cleared gradients, periodically and every epoch. It includes the model, optimizer, scheduler, scaler, RNG, deterministic sample order/cursor, partial epoch metrics, history and best state. Resume replays work since the last successful commit, not an arbitrary interrupt point. Per-sample augmentation depends only on seed/epoch/index, with zero loader workers. Same software/device/data are required for numerical reproducibility; CUDA bitwise equivalence is not promised. W&B may contain repeated progress values for replayed work.

First SIGINT requests a stop at the next safe boundary. A second interrupt or other exception saves separate **non-resumable emergency weights**, which may reflect a partial optimizer operation; it never intentionally replaces `last.pt`. Local commits survive Drive copy failures, which raise an explicit error and leave a `.sync-pending.pt` record. Re-run with `RESUME=True` after fixing storage. `best.pt` is a full checkpoint at an improved validation AUC; `last.pt` also contains the best weights.

**Current preset: five-epoch Bilateral fine-tuning.** Initialize from `raw_s42_recovered_20260910/raw_s42/best_weights.pt`, the best model of the latest completed raw phase, NOT the earlier emergency weights. Train only `bilateral_s42` for five additional epochs with LR 5e-5 in a separate `bilateral_s42_finetune5_from_raw_recovered` group. All three splits and both 2D views use Bilateral data. Missing source weights or incomplete denoised data fail before training; there is no raw/random-weight fallback. Optimizer and scheduler start fresh for this phase. Select best by Bilateral validation AUC, calibrate on Bilateral validation, then evaluate Bilateral test. XAI defaults off for real runs to avoid extra GPU work. To resume this phase later, set `RESUME=True` and clear both warm-start fields. Smoke exercises Bilateral preprocessing with tiny CPU data and no external checkpoint.

`FINAL_SMOKE=1` uses tiny synthetic CPU data and a separate tiny model, no dataset/pretrained downloads. W&B offline is explicitly permitted only for this local smoke verification; it does not test online W&B or Drive. Setup installs dependencies only for real Colab execution. Editing this notebook does not alter an already-running external kernel.

In [ ]:
import os, sys, subprocess
from pathlib import Path
SMOKE = os.environ.get('FINAL_SMOKE', '0') == '1'
if not SMOKE and 'google.colab' in sys.modules:
    repo = Path('/content/glaucoma-thesis')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Tqhuyen/glaucoma-thesis.git', str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm', 'wandb', 'scikit-image', 'scipy', 'matplotlib', 'huggingface_hub', 'python-dotenv'], check=True)
sys.path.insert(0, str(Path.cwd()))
import csv, json, random, tempfile, time
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
from scripts import final_model as fm, final_training as ft
ft.load_env_file()
DEVICE = torch.device('cpu' if SMOKE else ('cuda' if torch.cuda.is_available() else 'cpu'))
if SMOKE:
    torch.set_num_threads(1)


## Configuration
Use a distinct `RUN_GROUP` for a new experiment. `RESUME=True` requires matching config, data identities and saved W&B id. `RUN_TARGET` blank selects the full sweep; set it explicitly for recovery. `DENOISE_LIMIT` is the maximum additional samples processed per split per invocation; incomplete caches cannot enter training.

In [ ]:
RUN_GROUP = 'bilateral_s42_finetune5_from_raw_recovered'
RUN_TARGET = 'bilateral_s42'
RESUME = False
WARM_START_WEIGHTS = '' if SMOKE else '/content/drive/MyDrive/MasterBKDN/Thesis/final_2x2d_3d_crossgate/raw_s42_recovered_20260910/raw_s42/best_weights.pt'
WARM_START_TARGET = '' if SMOKE else RUN_TARGET
CHECKPOINT_EVERY_STEPS = 10
DATASETS = ['bilateral']
SEEDS = [42]
STORE_RES = 8 if SMOKE else 200
RES3D = 8 if SMOKE else 200
RES2D = 8 if SMOKE else 224
N_2D, D_LATENT, ENC2D = 2, 256, 'maxvit_tiny_rw_224'
EPOCHS, BS, GRAD_ACCUM = (1, 2, 2) if SMOKE else (5, 2, 8)
LR, WD, PATIENCE = 5e-5, 1e-4, 5
BUILD_DENOISED, DENOISE_METHOD, DENOISE_LIMIT = True, 'bilateral', 0
RUN_XAI = SMOKE
DENOISE_PARAMS = {'sigma_color': 0.10, 'sigma_spatial': 4.0}
DENOISE_IMPLEMENTATION = None
HF_RAW_REPO = 'tqhuyen/harvard-oct-glaucoma-200'
SPLITS = ('Training', 'Validation', 'Test')
SMOKE_ROOT = Path(tempfile.mkdtemp(prefix='final_smoke_')) if SMOKE else None
DATA_ROOT = SMOKE_ROOT / 'data' if SMOKE else Path('/content/final_data')
LOCAL_ROOT = SMOKE_ROOT / 'runs' if SMOKE else Path('outputs/final_2x2d_3d_crossgate') / RUN_GROUP
DRIVE_MOUNT = Path(os.environ.get('DRIVE_MOUNT', '/content/drive'))
DRIVE_ROOT = Path(os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/MasterBKDN/Thesis'))
DRIVE_DIR = DRIVE_ROOT / 'final_2x2d_3d_crossgate' / RUN_GROUP
selected = [(ds, seed, f'{ds}_s{seed}') for ds in DATASETS for seed in SEEDS if not RUN_TARGET or RUN_TARGET == f'{ds}_s{seed}']
if not selected:
    raise ValueError('RUN_TARGET does not match DATASETS/SEEDS')
if bool(WARM_START_WEIGHTS) != bool(WARM_START_TARGET):
    raise ValueError('Set both WARM_START_WEIGHTS and WARM_START_TARGET')
if WARM_START_WEIGHTS and (RESUME or RUN_TARGET != WARM_START_TARGET or len(selected) != 1):
    raise ValueError('Warm-start requires RESUME=False and one explicitly selected matching RUN_TARGET')
if not SMOKE:
    if not os.path.ismount(DRIVE_MOUNT):
        from google.colab import drive
        drive.mount(str(DRIVE_MOUNT))
    if not os.path.ismount(DRIVE_MOUNT) or not DRIVE_ROOT.resolve().is_relative_to(DRIVE_MOUNT.resolve()):
        raise RuntimeError('Drive must be a verified mounted filesystem, not a local directory')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
if WARM_START_WEIGHTS and not Path(WARM_START_WEIGHTS).is_file():
    raise FileNotFoundError(WARM_START_WEIGHTS)
if not SMOKE and (selected != [('bilateral', 42, 'bilateral_s42')] or EPOCHS != 5 or PATIENCE < EPOCHS or (not RESUME and not WARM_START_WEIGHTS)):
    raise ValueError('This preset requires five Bilateral epochs from existing weights, or resume of that phase')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
STORAGE = ft.Artifacts(LOCAL_ROOT, None if SMOKE else DRIVE_DIR, smoke=SMOKE)
DATA_STORAGE = ft.Artifacts(DATA_ROOT, None if SMOKE else DRIVE_DIR / 'data', smoke=SMOKE)
print('Device:', DEVICE, 'Local:', LOCAL_ROOT, 'Drive:', None if SMOKE else DRIVE_DIR)


## Data And Caches
Both 4D `(N,D,H,W)` and 5D `(N,1,D,H,W)` storage are indexed per sample. Raw 200-cubed arrays are never downsampled on disk. Denoising uses partial files plus a committed cursor and completion marker; legacy unmarked outputs are rebuilt rather than trusted. Views are keyed by their actual raw/denoised source and resolution and published only when complete. Cache identity uses path/size/mtime; training config additionally hashes complete volume/label files (cached SHA256). Identical raw data can resume after re-download. Do not edit files in place while preserving timestamps. Denoise partial caches are local recovery only; completed denoised arrays and generated views are synced before training.

In [ ]:
if SMOKE:
    rng = np.random.default_rng(0)
    for split, n in zip(SPLITS, (10, 6, 6)):
        np.save(DATA_ROOT / f'{split}_volumes.npy', rng.integers(0, 255, (n, 1, STORE_RES, STORE_RES, STORE_RES), dtype=np.uint8))
        np.save(DATA_ROOT / f'{split}_labels.npy', np.arange(n, dtype=np.int64) % 2)
else:
    from huggingface_hub import snapshot_download
    token = os.environ.get('HF_TOKEN')
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get('HF_TOKEN')
        except Exception:
            token = None
    if not all((DATA_ROOT / f'{s}_{kind}.npy').is_file() for s in SPLITS for kind in ('volumes', 'labels')):
        snapshot_download(repo_id=HF_RAW_REPO, repo_type='dataset', local_dir=str(DATA_ROOT), token=token)
if any(ds != 'raw' for ds, _, _ in selected):
    from scripts import compare_denoise_methods as cdm
    import skimage, scipy, hashlib
    if DENOISE_METHOD in ('bm3d', 'dncnn', 'swinir'):
        raise ValueError('This parameter-tracked path supports classical fixed-parameter denoisers only')
    if set(DENOISE_PARAMS) != set(cdm.METHODS[DENOISE_METHOD]['params']):
        raise ValueError('DENOISE_PARAMS must specify all and only parameters for the selected method')
    cdm.METHODS[DENOISE_METHOD]['params'] = dict(DENOISE_PARAMS)
    DENOISE_IMPLEMENTATION = {'module_sha256': hashlib.sha256(Path(cdm.__file__).read_bytes()).hexdigest(), 'skimage': skimage.__version__, 'scipy': scipy.__version__, 'numpy': np.__version__}
    if any(ds not in ('raw', DENOISE_METHOD) for ds, _, _ in selected):
        raise ValueError('Dataset tag must match DENOISE_METHOD')
    if BUILD_DENOISED:
        from scripts import compare_denoise_methods as cdm
        def denoise(volume):
            return cdm.denoise_volume(volume, DENOISE_METHOD, workers=2, cache_path=None)[0]
        complete = [ft.build_denoised(DATA_ROOT / f'{s}_volumes.npy', DATA_ROOT / f'{s}_volumes_dn.npy', denoise, method=DENOISE_METHOD, params=DENOISE_PARAMS, implementation=DENOISE_IMPLEMENTATION, limit=DENOISE_LIMIT) for s in SPLITS]
        if not all(complete):
            raise RuntimeError('Denoise incomplete; rerun data cell to continue. Training has not started.')
    for split in SPLITS:
        source = DATA_ROOT / f'{split}_volumes_dn.npy'
        if not source.with_suffix('.complete.pt').exists():
            raise RuntimeError('Denoised cache is not verified complete')
        marker = torch.load(source.with_suffix('.complete.pt'), weights_only=False)
        if marker.get('method') != DENOISE_METHOD or marker.get('params') != DENOISE_PARAMS or marker.get('implementation') != DENOISE_IMPLEMENTATION:
            raise ValueError('Completed denoise cache method/parameters/implementation mismatch')
        DATA_STORAGE.sync(source)
        DATA_STORAGE.sync(source.with_suffix('.complete.pt'))
def make_datasets(ds, seed):
    suffix = 'volumes' if ds == 'raw' else 'volumes_dn'
    datasets = [ft.FinalDataset(DATA_ROOT / f'{s}_{suffix}.npy', DATA_ROOT / f'{s}_labels.npy', res3d=RES3D, res2d=RES2D, seed=seed, train=s == 'Training') for s in SPLITS]
    if not SMOKE and any(tuple(d.volumes.shape[-3:]) != (200, 200, 200) for d in datasets):
        raise ValueError('Real training requires raw 200-cubed storage')
    if ds == 'bilateral' and any(d.source.name != f'{s}_volumes_dn.npy' for s, d in zip(SPLITS, datasets)):
        raise ValueError('Bilateral train/validation/test must all read denoised volumes')
    for split, dataset in zip(SPLITS, datasets):
        print(f'[input] {ds} {split}: {dataset.source} | views derived from this source')
    for split in SPLITS:
        for path in DATA_ROOT.glob(f'{split}_{suffix}_*{RES2D}*'):
            if '.partial.' not in path.name:
                DATA_STORAGE.sync(path)
    return datasets


## Train, Evaluate, Persist And Explain
`ACTIVE_TRAINER` and `ACTIVE_MODEL` remain accessible if training raises. Full state is checkpointed before the first batch; all errors propagate after best-effort emergency persistence and W&B finish. Results retain checkpoint paths and CPU predictions, not a sweep of GPU models. Calibration is fit on validation logits first, then the threshold is selected on calibrated validation probabilities and passed to both test metrics and bootstrap CI. Every saved report/XAI figure is synced immediately. XAI uses the first validation case, not a test-selected case.

In [ ]:
RESULTS = {}
ACTIVE_MODEL = ACTIVE_TRAINER = WANDB_RUN = None
for ds, seed, tag in selected:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    tr, va, te = make_datasets(ds, seed)
    ytr = tr.labels
    if set(np.unique(ytr)) != {0, 1}:
        raise ValueError('Training requires both binary classes')
    weights = [len(ytr) / (2 * int((ytr == c).sum())) for c in (0, 1)]
    config = dict(dataset=ds, seed=seed, epochs=EPOCHS, batch_size=BS, grad_accum=GRAD_ACCUM, lr=LR, weight_decay=WD, patience=PATIENCE, checkpoint_steps=CHECKPOINT_EVERY_STEPS, class_weights=weights, res3d=RES3D, res2d=RES2D, store_res=STORE_RES, n2d=N_2D, latent=D_LATENT, enc2d=ENC2D, smoke=SMOKE, torch_version=str(torch.__version__), device=str(DEVICE), denoise_method=DENOISE_METHOD if ds != 'raw' else 'none', data=[ft.data_identity(p) for d in (tr, va, te) for p in (d.source, d.label_path)])
    artifacts = ft.Artifacts(LOCAL_ROOT / tag, None if SMOKE else DRIVE_DIR / tag, smoke=SMOKE)
    config.update(denoise_params=DENOISE_PARAMS if ds != 'raw' else {}, denoise_implementation=DENOISE_IMPLEMENTATION if ds != 'raw' else None)
    config.update(run_xai=RUN_XAI, training_phase='bilateral_finetune5', parent_run='raw_s42_recovered_20260910/raw_s42')
    status = ft.run_status(artifacts, config, resume=RESUME)
    print(tag, status['status'])
    if status['status'] == 'complete':
        RESULTS[tag] = {'res': status['result']}
        continue
    tag_resume = status['status'] == 'resume'
    tag_warm_start = status['warm_start'] if status['status'] == 'initialized' else (WARM_START_WEIGHTS if tag == WARM_START_TARGET else '')
    if tag_warm_start and not Path(tag_warm_start).is_file():
        raise FileNotFoundError('Uncommitted warm-start requires its original weights: ' + tag_warm_start)
    WANDB_RUN = ft.init_wandb('final_' + tag, config, artifacts, resume=status['status'] != 'new', smoke=SMOKE, warm_start=tag_warm_start)
    exit_code, stopped = 1, False
    started = time.time()
    try:
        ACTIVE_MODEL = (ft.SmokeModel() if SMOKE else fm.FinalModel(n_2d=N_2D, D=D_LATENT, enc2d=ENC2D, enc2d_pretrained=not (tag_resume or bool(tag_warm_start)))).to(DEVICE)
        ACTIVE_TRAINER = ft.Trainer(ACTIVE_MODEL, tr, config, artifacts, WANDB_RUN, resume=tag_resume, warm_start=tag_warm_start)
        def evaluate(model):
            p, y, logits = ft.predict(model, va, BS)
            return {**fm.full_metrics(p, y), 'loss': float(torch.nn.functional.cross_entropy(torch.tensor(logits), torch.tensor(y)))}
        stopped = not ACTIVE_TRAINER.fit(evaluate)
        if stopped:
            WANDB_RUN.summary['stopped_safely'] = True
            exit_code = 0
        else:
            ACTIVE_MODEL.load_state_dict(ACTIVE_TRAINER.best_state)
            res, probs, labels = ft.calibrated_report(ACTIVE_MODEL, va, te, BS, smoke=SMOKE)
            res.update(tag=tag, seed=seed, hist=ACTIVE_TRAINER.history, minutes=round((time.time() - started) / 60, 2))
            WANDB_RUN.log({f'test/{k}': v for k, v in res['test'].items()})
            WANDB_RUN.log({f'val/calibrated_{k}': v for k, v in res['val'].items()})
            WANDB_RUN.summary.update({'threshold': res['threshold'], 'temperature': res['temperature'], **{f'test/{k}': v for k, v in res['test'].items()}})
            weights_path = artifacts.save(ft.cpu_state(ACTIVE_MODEL), 'best_weights.pt')
            ft.save_report(res, probs, labels, artifacts, WANDB_RUN)
            RESULTS[tag] = dict(res=res, weights_path=str(weights_path), test_probs=probs.tolist(), test_labels=labels.tolist())
            if RUN_XAI:
                ft.save_xai(ACTIVE_MODEL, va, artifacts, WANDB_RUN, smoke=SMOKE)
            exit_code = 0
    finally:
        WANDB_RUN.finish(exit_code=exit_code)
    if stopped:
        print('Stopped safely. Set RESUME=True and RUN_TARGET to', tag)
        break
    ft.complete_run(artifacts, config, WANDB_RUN.id)
    ACTIVE_MODEL.cpu()
    ACTIVE_TRAINER = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Completed:', list(RESULTS))


In [ ]:
rows = ft.publish_summary(STORAGE)
if rows:
    print(rows)
else:
    print('No completed runs; no summary or XAI selection attempted.')
print('Smoke verified only local/offline behavior.' if SMOKE else 'Completed artifacts were synced to the verified Drive mount.')
